# 02 — LRGB Peptides: real-data evidence

A reviewer for an IAPR/CIARP venue will want a result on real data, not only the synthetic NeighborsMatch.
The **Long Range Graph Benchmark** (Dwivedi et al., 2022) is the standard choice for long-range tasks.

- **Peptides-func**: multi-label graph classification, metric **AP** (average precision).
- **Peptides-struct**: graph regression, metric **MAE**.

We report QuotientNet against the same baselines, using the official splits and metric. The improvement may be
modest — the point is a *coherent real-data signal* consistent with the synthetic result.

> Requires `rdkit` (pulled by `environment.yml`). First run downloads the dataset to `../data/lrgb`.

In [ ]:
import torch, numpy as np, pandas as pd
from pathlib import Path
from oversquash.data import load_lrgb_peptides

NAME = 'Peptides-func'   # or 'Peptides-struct'
train_ds = load_lrgb_peptides(root='../data/lrgb', name=NAME, split='train')
val_ds   = load_lrgb_peptides(root='../data/lrgb', name=NAME, split='val')
test_ds  = load_lrgb_peptides(root='../data/lrgb', name=NAME, split='test')
print(NAME, '| train', len(train_ds), 'val', len(val_ds), 'test', len(test_ds))
print('example graph:', train_ds[0])

## Notes on adapting the quotient layer to molecular graphs

Unlike NeighborsMatch trees, peptide graphs have heterogeneous topology, so the QuotientPlan must be built
**per graph** (cache by a topology hash to avoid recomputing identical molecules). The walk→1-hop class
attribution (see `train.edge_classes_for_graph`) is heuristic on general graphs — document this in paper §3.

Day-5 task: wrap `build_quotient_plan` in a transform that attaches `edge_class` tensors to each `Data` object
at load time, so batching carries them along (this also fixes the batched-edge-class issue from notebook 01).

In [ ]:
# Skeleton: per-graph plan precomputation (cache by topology). Fill in day 5.
from oversquash.ideal_bridge import build_quotient_plan
from torch_geometric.transforms import BaseTransform

class AttachEdgeClasses(BaseTransform):
    """Attach per-depth edge-class ids (under I) to each Data object.
    Caches by (num_nodes, sorted-edge-tuple) so identical topologies are built once."""
    def __init__(self, n_layers):
        self.n_layers = n_layers
        self._cache = {}
    def __call__(self, data):
        ei = data.edge_index.numpy()
        key = (data.num_nodes, tuple(map(tuple, ei.T.tolist())))
        if key not in self._cache:
            # TODO day 5: derive per-depth class tensors from the plan, like
            # train.edge_classes_for_graph, but returned as a stacked tensor.
            _ = build_quotient_plan(ei, data.num_nodes, max_length=self.n_layers)
            self._cache[key] = None  # placeholder
        data.edge_classes = self._cache[key]
        return data

print('AttachEdgeClasses defined — wire into LRGBDataset(transform=...) on day 5.')

## Train + evaluate (day 5–6)

Use the official AP (func) / MAE (struct) metric from PyG, the official splits above, and the same width/depth
as the baselines. Save the results table to `../results/tables/lrgb_peptides.csv`.

In [ ]:
# Placeholder for the training cell — depends on AttachEdgeClasses (day 5).
# Baselines (GCN/GAT/GIN) can already be trained here today without edge classes
# to establish the reference numbers; add QuotientNet once the transform is done.
print('TODO day 5–6: train baselines + QuotientNet on', NAME)